# Səhv Detektivi: v3 adapterini vLLM ilə qaldır və masterai-yə qoş

Adapter `MyDrive/sehv-detektivi/sehv-lora-v3`-dədir (`04_v3_telim.ipynb`). Bu notebook:
1. vLLM-i `--lora-modules sehv=...` ilə qaldırır (qəbz adapteri Drive-da varsa, o da eyni serverə qoşulur);
2. golden dəsti **masterai-nin öz yolu** ilə ölçür (`python -m evals.sehv`): yalnız sympy, yalnız model, model+sympy;
3. Cloudflare tuneli açır və serverin `.env`-inə yazılacaq sətirləri çap edir.

**Runtime → T4 GPU → Run all.** Yeni sessiyada işlət (vLLM öz torch versiyasını qoyur).
Colab bağlananda tunel ölür: server riyaziyyat üçün sympy-yə düşür, məntiq həllinə «model əlçatan deyil» deyir.

In [ ]:
# 1) vLLM + repo + Drive
!pip -q install "vllm==0.6.6.post1" httpx sympy
!rm -rf masterai && git clone -q --depth 1 https://github.com/Alihuseyn-ibrahimov/masterai.git
import os, re, secrets, subprocess, sys, time, urllib.request
sys.path.insert(0, "/content/masterai")
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/sehv-detektivi"
ADAPTER = f"{DRIVE}/sehv-lora-v3"
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "Adapter tapılmadı: əvvəl 04_v3_telim.ipynb işlət"
RECEIPTS = "/content/drive/MyDrive/masterai-receipts/adapter"
modules = [f"sehv={ADAPTER}"]
if os.path.exists(f"{RECEIPTS}/adapter_config.json"):
    modules.append(f"receipts={RECEIPTS}")
print("LoRA modulları:", modules)

In [ ]:
# 2) Server (baza model təlimdəki ilə eynidir)
API_KEY = secrets.token_urlsafe(24)
log = open("vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct", "--dtype", "half", "--max-model-len", "2048",
    "--enable-lora", "--lora-modules", *modules, "--max-lora-rank", "16", "--max-loras", str(len(modules)),
    "--gpu-memory-utilization", "0.85", "--api-key", API_KEY, "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("vLLM hazırdır"); break
    except Exception:
        time.sleep(5)
else:
    print(open("vllm.log").read()[-3000:])

## Golden dəst masterai yolu ilə
Səhv Detektivi reposundakı `evaluate.py` modelin özünü ölçür; bu isə tələbənin gördüyü nəticəni:
eyni prompt (`pipeline.sehv_detektivi.prompts`, test hash ilə qoruyur), parse, sympy vetosu və
«yoxlanmadı» halı. v3 golden (HF generate): addım 0.69, var/yox 0.76, yalançı həyəcan 0.41.
vLLM rəqəmləri ondan bir az fərqlənə bilər (fp16, başqa nüvələr): fərq böyükdürsə səbəb axtarılmalıdır.

In [ ]:
!cd /content/masterai && python -m evals.sehv --url http://localhost:8000 --model sehv --api-key {API_KEY} --out {DRIVE}/eval_masterai_v3.md

## Bir nümunə

In [ ]:
from pipeline.sehv_detektivi import VllmBackend, diagnose
backend = VllmBackend("http://localhost:8000", "sehv", api_key=API_KEY, timeout=60)
diagnose("mentiq", "Bütün pişiklər heyvandır. Mırtı heyvandır. Mırtı pişikdirmi?",
         ["Bütün pişiklər heyvandır", "Mırtı heyvandır", "Deməli, Mırtı pişikdir"], backend)

## MasterAI serverinə qoş
Serverdə `/opt/masterai/.env`-ə yaz və `systemctl restart masterai`. Qəbz adapteri də qoşulubsa,
`VLLM_RECEIPT_MODEL=receipts` sətri də çap olunur (eyni ünvan və açar).

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if match:
        url = match.group(0); break
lines = [f"VLLM_BASE_URL={url}", f"VLLM_API_KEY={API_KEY}", "SEHV_VLLM_MODEL=sehv"]
if len(modules) > 1:
    lines.append("VLLM_RECEIPT_MODEL=receipts")
print("\n".join(lines))
print("\nYoxlama: restartdan sonra «Həllimi yoxla» səhifəsində «AI model hazırda qoşulmayıb» yazısı görünməməlidir.")